In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

In [ ]:
pd.__version__

In [ ]:
np.__version__

In [ ]:
#!pip install jupyter-black

In [2]:
%load_ext jupyter_black

In [3]:
df = pd.read_csv("../../data/intermediate/data_concat.csv", header=0)

/tmp/ipykernel_26561/2374769283.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/intermediate/data_concat.csv", header=0)


In [ ]:
def df_initial_preproc(df):
    df.month = pd.to_datetime(df.month)
    df["year_of_sales"] = df["month"].dt.year
    df["month_of_sales"] = df["month"].dt.month
    # Clean up the 'MULTI-GENERATION' entries
    df["flat_type"] = df["flat_type"].str.replace(
        "MULTI GENERATION", "MULTI-GENERATION"
    )
    # add price per sqm
    df["price_per_sqm"] = df.resale_price / df.floor_area_sqm
    return df

# Price Per Square meter

In [ ]:
# price per sqm across different town
def plot_sqm_all_town(df):
    df_initial_preproc(df)
    sns.set_style("whitegrid")
    sns.set_palette("bright")
    g = sns.catplot(
        data=df,
        x="price_per_sqm",
        y="town",
        kind="bar",
        height=7,
        aspect=1,
        errorbar=None,
    )
    g.fig.suptitle("Price Per Square Meter across different town", y=1.02, fontsize=15)
    g.set(xlabel="Town", ylabel="Price Per Square Meter")

    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=14
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=14
    )  # Access and set y-axis label font size

    plt.xticks(rotation=0)
    plt.show()

In [ ]:
plt.clf()
plot_sqm_all_town(df)

In [ ]:
# price per sqm across single town and flat type
def plot_sqm_single_twn_room(df, room, twn):
    df_initial_preproc(df)
    df_query = df.query("flat_type == @room & town == @twn")
    sns.set_style("whitegrid")
    sns.set_palette("bright")

    g = sns.catplot(
        data=df_query,
        x="year_of_sales",
        y="price_per_sqm",
        kind="bar",
        height=5,
        aspect=2.5,
        errorbar=None,
    )
    g.fig.suptitle(
        f"Price Per Square Meter across {twn} and flat type: {room}",
        y=1.01,
        fontsize=18,
    )
    g.set(xlabel="Year of sales", ylabel="Price Per Square Meter")
    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=16
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=16
    )  # Access and set y-axis label font size
    plt.xticks(rotation=45)

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_sqm_single_twn_room(df, "4 ROOM", "PUNGGOL")

## Resale price vs Town and flat type

In [ ]:
import matplotlib.ticker as mtick

In [ ]:
# mean resale price across different town
def plot_resale_price_all(df):
    df_initial_preproc(df)
    sns.set_style("whitegrid")
    sns.set_palette("bright")

    # Calculate the mean resale price for each town
    mean_prices = df.groupby("town")["resale_price"].mean().sort_values()

    # Create a new categorical order based on the sorted mean prices
    town_order = mean_prices.index.tolist()

    g = sns.catplot(
        data=df,
        x="town",
        y="resale_price",
        kind="bar",
        height=5,
        aspect=2.5,
        errorbar=None,
        order=town_order,
    )
    g.fig.suptitle("Mean Resale Price across different town", y=1.01, fontsize=16)
    g.set(xlabel="Town", ylabel="Resale Price")
    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=15
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=15
    )  # Access and set y-axis label font size
    plt.xticks(rotation=90)

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_resale_price_all(df)

In [ ]:
# mean resale price across single town and different flat type
def plot_resale_price_single(twn):
    df_initial_preproc(df)
    df_query = df.query("town == @twn")
    plt.clf()
    sns.set_style("whitegrid")
    sns.set_palette("bright")
    hue_order = [
        "1 ROOM",
        "2 ROOM",
        "3 ROOM",
        "4 ROOM",
        "5 ROOM",
        "EXECUTIVE",
        "MULTI-GENERATION",
    ]
    g = sns.catplot(
        data=df_query,
        x="town",
        y="resale_price",
        kind="bar",
        height=5,
        aspect=2,
        errorbar=None,
        hue="flat_type",
        hue_order=hue_order,
        palette="bright",
    )
    g.fig.suptitle(
        f"Mean Resale Price across {twn} and different flat type", y=1.01, fontsize=16
    )
    g.set(xlabel="Town", ylabel="Resale Price")

    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=15
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=15
    )  # Access and set y-axis label font size

    plt.xticks(rotation=0)

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_resale_price_single("ANG MO KIO")

In [ ]:
def data_resale_price_single(df, room, twn):
    df_initial_preproc(df)
    df_query = df.query("flat_type == @room & town == @twn")
    price_summary_series = df_query["resale_price"].agg(["max", "min", "mean"])
    # Convert the Series to a DataFrame
    price_summary_df = (
        (
            price_summary_series.to_frame(
                name=f"Resale price summary of {twn}, {room} flat"
            )
        )
        .round()
        .astype(int)
    )

    # Apply the formatting to the column
    price_summary_df[price_summary_df.columns[0]] = price_summary_df[
        price_summary_df.columns[0]
    ].apply(lambda x: "{:,}".format(x))

    return price_summary_df

In [ ]:
data_resale_price_single(df, "4 ROOM", "PUNGGOL")

## Resale Price VS month

In [ ]:
# mean resale price per month across all town
def plot_pricePerMonth_all(df):
    df_initial_preproc(df)
    sns.set_style("ticks")
    sns.set_palette("bright")
    # hue_order = ["1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM","EXECUTIVE", "MULTI-GENERATION"]
    g = sns.catplot(
        data=df,
        x="month_of_sales",
        y="resale_price",
        kind="bar",
        height=5,
        aspect=2,
        errorbar=None,
    )
    g.fig.suptitle(
        "Mean resale Price per month across different town and flat type",
        y=1.01,
        fontsize=16,
    )
    g.set(xlabel="Month of Sales", ylabel="Resale Price")

    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=15
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=15
    )  # Access and set y-axis label font size

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_pricePerMonth_all(df)

In [ ]:
# mean resale price per month across single town
def plot_pricePerMonth_single(df, room, twn):
    df_initial_preproc(df)
    df_query = df.query("flat_type == @room & town == @twn")
    sns.set_style("ticks")
    sns.set_palette("bright")
    # hue_order = ["1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM","EXECUTIVE", "MULTI-GENERATION"]
    g = sns.catplot(
        data=df,
        x="month_of_sales",
        y="resale_price",
        kind="bar",
        height=5,
        aspect=1.5,
        errorbar=None,
    )
    g.fig.suptitle(
        f"Resale Price per month across {twn} and flat type: {room}",
        y=1.01,
        fontsize=15,
    )
    g.set(xlabel="Month of Sales", ylabel="Resale Price")
    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=14
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=14
    )  # Access and set y-axis label font size

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)
    plt.show()

In [ ]:
plt.clf()
plot_pricePerMonth_single(df, "4 ROOM", "PUNGGOL")

## Resale price over the years

In [ ]:
# resale price trend across all town
def plot_priceTrend_all(df):
    df_initial_preproc(df)
    sns.set_style("ticks")
    sns.set_palette("bright")
    hue_order = [
        "1 ROOM",
        "2 ROOM",
        "3 ROOM",
        "4 ROOM",
        "5 ROOM",
        "EXECUTIVE",
        "MULTI-GENERATION",
    ]
    g = sns.relplot(
        data=df,
        x="month",
        y="resale_price",
        kind="line",
        height=5,
        aspect=2,
        palette="bright",
        errorbar=None,
        hue="flat_type",
        hue_order=hue_order,
    )
    g.fig.suptitle(f"Resale price trend across all town", y=1.01, fontsize=17)
    g.set(xlabel="Year", ylabel="Resale Price")

    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=16
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=16
    )  # Access and set y-axis label font size

    plt.ticklabel_format(style="plain", axis="y")

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_priceTrend_all(df)

In [ ]:
# resale price trend across single town and flat type
def plot_priceTrend_single(df, room, twn):
    df_initial_preproc(df)
    df_query = df.query("flat_type == @room & town == @twn")
    sns.set_style("ticks")
    sns.set_palette("bright")
    g = sns.relplot(
        data=df_query,
        x="month",
        y="resale_price",
        kind="line",
        height=5,
        aspect=2,
        errorbar=None,
    )
    g.fig.suptitle(
        f"Resale price trend across {twn} and flat type: {room}", y=1.01, fontsize=16
    )
    g.set(xlabel="Year", ylabel="Resale Price")
    g.ax.set_xlabel(
        g.ax.get_xlabel(), fontsize=15
    )  # Access and set x-axis label font size

    g.ax.set_ylabel(
        g.ax.get_ylabel(), fontsize=15
    )  # Access and set y-axis label font size

    # Format the y-axis tick labels to include commas using a lambda function
    formatter = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}")
    g.ax.yaxis.set_major_formatter(formatter)

    plt.show()

In [ ]:
plt.clf()
plot_priceTrend_single(df, "4 ROOM", "PUNGGOL")

In [ ]:
df = df_initial_preproc(df)

In [ ]:
df.head()

In [ ]:
df_query = df.query("flat_type == '4 ROOM' & town == 'PUNGGOL'")
df_last_resale_price = df_query.sort_index(ascending=False)
last_resale_price = df_last_resale_price.iloc[0]["resale_price"]
formatted_price = "{:,}".format(int(round(last_resale_price)))
formatted_price

In [ ]:
def data_last_resale_price(df, room, twn):
    # df_initial_preproc(df)
    df_query = df.query("flat_type == @room & town == @twn")

    df_last_resale_price = df_query.sort_index(ascending=False)
    last_resale_price = df_last_resale_price.iloc[0]["resale_price"]

    formatted_price = "{:,}".format(int(round(last_resale_price)))

    return formatted_price

In [ ]:
data_last_resale_price(df, "4 ROOM", "PUNGGOL")

In [ ]:
#changing town and flat type from capital letter to lower capital letter

df = df_initial_preproc(df)

In [ ]:
df.head()

In [ ]:
df.town.unique()

In [ ]:
town_mapping = {
    "TAMPINES": "Tampines",
    "YISHUN": "Yishun",
    "JURONG WEST": "Jurong West",
    "BEDOK": "Bedok",
    "WOODLANDS": "Woodlands",
    "ANG MO KIO": "Ang Mo Kio",
    "HOUGANG": "Hougang",
    "BUKIT BATOK": "Bukit Batok",
    "CHOA CHU KANG": "Choa Chu Kang",
    "BUKIT MERAH": "Bukit Merah",
    "SENGKANG": "Sengkang",
    "PASIR RIS": "Pasir Ris",
    "TOA PAYOH": "Toa Payoh",
    "QUEENSTOWN": "Queenstown",
    "GEYLANG": "Geylang",
    "CLEMENTI": "Clementi",
    "BUKIT PANJANG": "Bukit Panjang",
    "KALLANG/WHAMPOA": "Kallang_Whampoa",
    "JURONG EAST": "Jurong East",
    "SERANGOON": "Serangoon",
    "PUNGGOL": "Punggol",
    "BISHAN": "Bishan",
    "SEMBAWANG": "Sembawang",
    "MARINE PARADE": "Marine Parade",
    "CENTRAL AREA": "Central Area",
    "BUKIT TIMAH": "Bukit Timah",
    "LIM CHU KANG": "Lim Chu Kang",
}
df["town_map"] = df["town"].map(town_mapping)

In [ ]:
df.head()

In [ ]:
df.flat_type.unique()

In [ ]:
room_mapping = {
    "1 ROOM": "1 room",
    "2 ROOM": "2 room",
    "3 ROOM": "3 room",
    "4 ROOM": "4 room",
    "5 ROOM": "5 room",
    "EXECUTIVE": "Executive",
    "MULTI-GENERATION": "Multi-Gen",
}
df["flat_type_map"] = df["flat_type"].map(room_mapping)

In [ ]:
df.drop(["town", "flat_type"], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.rename(columns={"town_map": "town", "flat_type_map": "flat_type"}, inplace=True)

In [ ]:
df.head()

In [4]:
def df_initial_preproc(df):
    town_mapping = {
        "TAMPINES": "Tampines",
        "YISHUN": "Yishun",
        "JURONG WEST": "Jurong West",
        "BEDOK": "Bedok",
        "WOODLANDS": "Woodlands",
        "ANG MO KIO": "Ang Mo Kio",
        "HOUGANG": "Hougang",
        "BUKIT BATOK": "Bukit Batok",
        "CHOA CHU KANG": "Choa Chu Kang",
        "BUKIT MERAH": "Bukit Merah",
        "SENGKANG": "Sengkang",
        "PASIR RIS": "Pasir Ris",
        "TOA PAYOH": "Toa Payoh",
        "QUEENSTOWN": "Queenstown",
        "GEYLANG": "Geylang",
        "CLEMENTI": "Clementi",
        "BUKIT PANJANG": "Bukit Panjang",
        "KALLANG/WHAMPOA": "Kallang_Whampoa",
        "JURONG EAST": "Jurong East",
        "SERANGOON": "Serangoon",
        "PUNGGOL": "Punggol",
        "BISHAN": "Bishan",
        "SEMBAWANG": "Sembawang",
        "MARINE PARADE": "Marine Parade",
        "CENTRAL AREA": "Central Area",
        "BUKIT TIMAH": "Bukit Timah",
        "LIM CHU KANG": "Lim Chu Kang",
    }

    room_mapping = {
        "1 ROOM": "1 room",
        "2 ROOM": "2 room",
        "3 ROOM": "3 room",
        "4 ROOM": "4 room",
        "5 ROOM": "5 room",
        "EXECUTIVE": "Executive",
        "MULTI-GENERATION": "Multi-Gen",
    }

    df.month = pd.to_datetime(df.month)
    df["year_of_sales"] = df["month"].dt.year
    df["month_of_sales"] = df["month"].dt.month
    # Clean up the 'MULTI-GENERATION' entries
    df["flat_type"] = df["flat_type"].str.replace(
        "MULTI GENERATION", "MULTI-GENERATION"
    )
    # add price per sqm
    df["price_per_sqm"] = df.resale_price / df.floor_area_sqm

    # change town and flat_type name to lower cap
    df["town_map"] = df["town"].map(town_mapping)
    df["flat_type_map"] = df["flat_type"].map(room_mapping)

    df.drop(["town", "flat_type"], axis=1, inplace=True)
    df.rename(columns={"town_map": "town", "flat_type_map": "flat_type"}, inplace=True)

    return df

In [5]:
df_initial_preproc(df)

,month,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,year,year_of_sales,month_of_sales,price_per_sqm,town,flat_type
0,1990-01-01,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,9000.0,NaN,1990,1990,1,290.322581,Ang Mo Kio,1 room
1,1990-01-01,44,BENDEMEER RD,04 TO 06,63.0,STANDARD,1981,31400.0,NaN,1990,1990,1,498.412698,Kallang_Whampoa,3 room
2,1990-01-01,20,ST. GEORGE'S RD,04 TO 06,67.0,NEW GENERATION,1984,66500.0,NaN,1990,1990,1,992.537313,Kallang_Whampoa,3 room
3,1990-01-01,14,KG ARANG RD,04 TO 06,103.0,NEW GENERATION,1984,77000.0,NaN,1990,1990,1,747.572816,Kallang_Whampoa,3 room
4,1990-01-01,46,OWEN RD,01 TO 03,68.0,NEW GENERATION,1982,58000.0,NaN,1990,1990,1,852.941176,Kallang_Whampoa,3 room
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
949506,2025-03-01,613A,TAMPINES NTH DR 1,10 TO 12,93.0,Model A,2020,795000.0,94 years 08 months,2025,2025,3,8548.387097,Tampines,4 room
949507,2025-03-01,608A,TAMPINES NTH DR 1,04 TO 06,93.0,Model A,2020,715000.0,94 years 08 months,2025,2025,3,7688.172043,Tampines,4 room
949508,2025-03-01,609A,TAMPINES NTH DR 1,10 TO 12,93.0,Model A,2020,790000.0,94 years 08 months,2025,2025,3,8494.623656,Tampines,4 room
949509,2025-03-01,113,TAMPINES ST 11,04 TO 06,115.0,Model A,1982,700000.0,56 years 05 months,2025,2025,3,6086.956522,Tampines,4 room
